# Task 4 — VCF → PubMed Search
Read VCF variants and search PubMed for related publications.

## 1. Initialize Project Environment
Import dependencies for VCF parsing and PubMed queries.

In [19]:
from __future__ import annotations

import logging
from pathlib import Path
from typing import Dict, List, Tuple

import pandas as pd
from Bio import Entrez

logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s")

print("pandas", pd.__version__)
try:
    import Bio

    print("biopython", Bio.__version__)
except Exception as exc:
    logging.error("Biopython import failed: %s", exc)

pandas 2.2.3
biopython 1.85


## 2. Define Configuration Parameters

In [20]:
from dataclasses import dataclass, asdict


@dataclass
class VcfPubmedConfig:
    handle: str
    email: str = "student@example.com"
    max_articles_per_variant: int = 3
    export_dir: Path = Path("artifacts")

    def describe(self) -> Dict[str, str]:
        info = asdict(self)
        info["export_dir"] = str(info["export_dir"])
        return info


CONFIG = VcfPubmedConfig(handle="AndreiCod")
CONFIG.describe()

{'handle': 'AndreiCod',
 'email': 'student@example.com',
 'max_articles_per_variant': 3,
 'export_dir': 'artifacts'}

In [21]:
CONFIG.export_dir.mkdir(parents=True, exist_ok=True)
VCF_FILE = CONFIG.export_dir / "task4_tp53_variants.vcf"

vcf_content = """##fileformat=VCFv4.2
##source=lab03_assignment
##reference=GRCh38
#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO
17\t7676154\trs1042522\tC\tG\t.\tPASS\tNOTE=TP53_Pro72Arg
17\t7674220\trs28934578\tC\tT\t.\tPASS\tNOTE=TP53_R175H
17\t7673802\t.\tG\tA\t.\tPASS\tNOTE=TP53_exon7
"""

with open(VCF_FILE, "w") as f:
    f.write(vcf_content)
print(f"Created VCF: {VCF_FILE}")

Created VCF: artifacts/task4_tp53_variants.vcf


## 3. Implement Core Functionality

In [22]:
def parse_vcf(vcf_path: Path) -> List[Dict]:
    variants = []
    with open(vcf_path, "r") as f:
        for line in f:
            if line.startswith("#"):
                continue
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            chrom, pos, var_id, ref, alt = parts[:5]
            info = parts[7] if len(parts) > 7 else ""
            variants.append(
                {
                    "chrom": chrom,
                    "pos": pos,
                    "id": var_id if var_id != "." else None,
                    "ref": ref,
                    "alt": alt,
                    "info": info,
                }
            )
    return variants


variants = parse_vcf(VCF_FILE)
print(f"Found {len(variants)} variant(s)")
pd.DataFrame(variants)

Found 3 variant(s)


,chrom,pos,id,ref,alt,info
0,17,7676154,rs1042522,C,G,NOTE=TP53_Pro72Arg
1,17,7674220,rs28934578,C,T,NOTE=TP53_R175H
2,17,7673802,None,G,A,NOTE=TP53_exon7


In [23]:
def build_pubmed_query(variant: Dict) -> str:
    if variant["id"] and variant["id"].startswith("rs"):
        return variant["id"]
    return f"chr{variant['chrom']}:{variant['pos']} AND TP53"


def search_pubmed_for_variant(
    variant: Dict, max_results: int, email: str
) -> Tuple[List[Dict], str]:
    Entrez.email = email
    query = build_pubmed_query(variant)
    handle = Entrez.esearch(db="pubmed", term=query, retmax=max_results)
    record = Entrez.read(handle)
    handle.close()
    pmids = record["IdList"]
    articles = []
    for pmid in pmids:
        handle = Entrez.efetch(db="pubmed", id=pmid, rettype="xml", retmode="xml")
        records = Entrez.read(handle)
        handle.close()
        if records["PubmedArticle"]:
            article = records["PubmedArticle"][0]["MedlineCitation"]["Article"]
            title = article.get("ArticleTitle", "No title")
            authors = [
                a["LastName"]
                for a in article.get("AuthorList", [])[:3]
                if "LastName" in a
            ]
            articles.append({"pmid": pmid, "title": title, "authors": authors})
    return articles, query


print("Functions defined.")

Functions defined.


In [24]:
all_results = []
for i, variant in enumerate(variants, 1):
    print(
        f"Processing variant {i}/{len(variants)}: Chr{variant['chrom']}:{variant['pos']}"
    )
    articles, query = search_pubmed_for_variant(
        variant, CONFIG.max_articles_per_variant, CONFIG.email
    )
    print(f"  Query: '{query}' -> {len(articles)} articles")
    all_results.append({"variant": variant, "query": query, "articles": articles})

Processing variant 1/3: Chr17:7676154


  Query: 'rs1042522' -> 3 articles
Processing variant 2/3: Chr17:7674220
  Query: 'rs28934578' -> 1 articles
Processing variant 3/3: Chr17:7673802
  Query: 'chr17:7673802 AND TP53' -> 3 articles


In [25]:
summary = [
    {
        "Variant": f"chr{r['variant']['chrom']}:{r['variant']['pos']}",
        "rsID": r["variant"]["id"] or "N/A",
        "Query": r["query"],
        "Articles": len(r["articles"]),
    }
    for r in all_results
]
pd.DataFrame(summary)

,Variant,rsID,Query,Articles
0,chr17:7676154,rs1042522,rs1042522,3
1,chr17:7674220,rs28934578,rs28934578,1
2,chr17:7673802,N/A,chr17:7673802 AND TP53,3


## 4. Validate with Unit Tests

In [26]:
def test_parse_vcf():
    assert len(variants) >= 2


def test_query_building():
    assert (
        build_pubmed_query({"id": "rs1042522", "chrom": "17", "pos": "7676154"})
        == "rs1042522"
    )
    assert "chr17:7673802" in build_pubmed_query(
        {"id": None, "chrom": "17", "pos": "7673802"}
    )


test_parse_vcf()
test_query_building()
print("All tests passed.")

All tests passed.


## 5. Export Results

In [27]:
out_file = CONFIG.export_dir / "task4_vcf_pubmed_results.txt"
with open(out_file, "w", encoding="utf-8") as f:
    f.write(
        f"VCF -> PubMed Search Results\nSource VCF: {VCF_FILE.name}\n"
        + "=" * 80
        + "\n\n"
    )
    for i, result in enumerate(all_results, 1):
        v = result["variant"]
        f.write(f"Variant {i}\n" + "-" * 40 + "\n")
        f.write(
            f"Position: chr{v['chrom']}:{v['pos']}\nRef/Alt: {v['ref']} > {v['alt']}\n"
        )
        f.write(
            f"ID: {v['id'] or 'N/A'}\nPubMed Query: {result['query']}\nArticles found: {len(result['articles'])}\n\n"
        )
        for j, art in enumerate(result["articles"], 1):
            f.write(f"  [{j}] PMID: {art['pmid']}\n      Title: {art['title']}\n\n")
        f.write("=" * 80 + "\n\n")
print(f"[OK] Results saved to: {out_file.resolve()}")

[OK] Results saved to: /home/rbals/git/daha-bdhb/BDHB-lab/labs/03_formats&NGS/assignments/artifacts/task4_vcf_pubmed_results.txt
